# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [ ]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
%pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel

from tqdm.auto import tqdm

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR = Path("/content/drive/MyDrive/final_dl")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
# Higher resolution helps ScienceQA diagrams, maps, axes, and embedded text.
IMG_SIZE = 384

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load and Preprocess Data

In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

In [ ]:
print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

In [ ]:

CHOICE_LETTERS = "ABCDEFGHIJ"
USE_METADATA_IN_PROMPT = True
PROMPT_METADATA_COLUMNS = ["subject", "grade", "topic"]
ANSWER_PREFIX = "The correct answer is:"


def format_prompt_metadata(row: pd.Series) -> str:
    if not USE_METADATA_IN_PROMPT:
        return ""

    label_map = {
        "subject": "Subject",
        "grade": "Grade",
        "topic": "Topic",
    }
    metadata_lines = []

    for col in PROMPT_METADATA_COLUMNS:
        value = row.get(col, "")
        if pd.notna(value) and str(value).strip():
            label = label_map.get(col, col.replace("_", " ").title())
            metadata_lines.append(f"{label}: {str(value).strip()}")

    return "\n".join(metadata_lines)


def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []
    lecture = row.get("lecture", "")
    hint = row.get("hint", "")

    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    metadata_str = format_prompt_metadata(row)
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    prompt += "Use the image and the provided information to answer the multiple-choice question.\n\n"
    if metadata_str:
        prompt += f"Metadata:\n{metadata_str}\n\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n\n"
    prompt += "Reply with only one letter and nothing else.\n"
    prompt += ANSWER_PREFIX

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

## 3. Model Training

In [ ]:
%pip install -q transformers peft accelerate datasets

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Load processor
processor = AutoProcessor.from_pretrained(MODEL_ID)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# Load model
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

# Gradient checkpointing trades compute for memory, enabling 384x384 training.
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# Manually move model to device when running on CPU
if not torch.cuda.is_available():
    model.to(device)

# Stay in train mode before training; Trainer will handle train/eval switching.
model.train()

print("Processor loaded.")
print("Model loaded.")
print("Gradient checkpointing enabled:", getattr(model, "is_gradient_checkpointing", "unknown"))
print("Using device:", model.device)

In [ ]:
USE_DORA = True

# Attention + MLP LoRA at lower rank. DoRA improves adapter quality by learning
# weight magnitude separately from direction, while staying within the 5M budget.
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "v_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_dora=USE_DORA,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"DoRA enabled: {USE_DORA}")
print(f"Trainable params budget check: {trainable_params:,} / 5,000,000")
if trainable_params > 5_000_000:
    raise ValueError("LoRA/DoRA trainable parameter count exceeds the 5M budget. Lower r or reduce target_modules.")

In [ ]:
class VQATrainDataset(Dataset):
    def __init__(self, df, processor, data_dir, img_size):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.data_dir = data_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(self.data_dir / row["image_path"]).convert("RGB").resize((self.img_size, self.img_size))

        prefix_text = build_prompt(row, include_answer=False)
        full_text = build_prompt(row, include_answer=True)

        enc = self.processor(
            text=[full_text],
            images=[image],
            return_tensors="pt",
            padding=False,
            truncation=False,
        )

        prefix_enc = self.processor(
            text=[prefix_text],
            images=[image],
            return_tensors="pt",
            padding=False,
            truncation=False,
        )

        item = {}
        for k, v in enc.items():
            item[k] = v.squeeze(0)

        labels = item["input_ids"].clone()

        prefix_len = prefix_enc["input_ids"].shape[1]

        # The prompt prefix does not contribute to the loss
        labels[:prefix_len] = -100

        item["labels"] = labels
        return item

In [ ]:
def multimodal_collate_fn(batch):
    input_ids = [x["input_ids"] for x in batch]
    attention_mask = [x["attention_mask"] for x in batch]
    pixel_values = [x["pixel_values"] for x in batch]
    labels = [x["labels"] for x in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id,
    )

    attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_mask,
        batch_first=True,
        padding_value=0,
    )

    labels = torch.nn.utils.rnn.pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100,
    )

    pixel_values = torch.stack(pixel_values, dim=0)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pixel_values": pixel_values,
        "labels": labels,
    }

In [ ]:
# train_small = train_df.sample(min(1000, len(train_df)), random_state=42).reset_index(drop=True)
# val_small = val_df.sample(min(200, len(val_df)), random_state=42).reset_index(drop=True)

train_full = train_df.reset_index(drop=True)
train_dataset = VQATrainDataset(train_full, processor, DATA_DIR, IMG_SIZE)


In [ ]:
training_args = TrainingArguments(
    output_dir="./smolvlm_lora_out",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=200,
    eval_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=multimodal_collate_fn,
)

trainer.train()

In [ ]:
trainer.save_model("./smolvlm_lora_out/final")
processor.save_pretrained("./smolvlm_lora_out/final")
print("saved.")

In [ ]:
SAVE_DIR = Path("/content/drive/MyDrive/final_dl/my_trained_model")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SAVE_DIR))
processor.save_pretrained(str(SAVE_DIR))

print("Saved to:", SAVE_DIR)

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
SAVE_DIR = "/content/drive/MyDrive/final_dl/my_trained_model"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(SAVE_DIR)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base_model, SAVE_DIR)

if not torch.cuda.is_available():
    model.to(device)

model.eval()

print("Loaded trained model from:", SAVE_DIR)
print("Using device:", model.device)

In [ ]:
# Evaluate on the val set using multiple-choice log-likelihood
CHOICE_LETTERS = "ABCDEFGHIJ"
ANALYSIS_COLUMNS = [
    "num_choices",
    "task",
    "grade",
    "subject",
    "topic",
    "category",
    "skill",
]

# Ensure candidates within a batch are right-padded, so answer tokens can be sliced out with prefix_len.
processor.tokenizer.padding_side = "right"


def get_model_device(model):
    model_device = getattr(model, "device", None)
    if model_device is not None:
        return model_device
    return next(model.parameters()).device


def move_to_model_device(inputs, model):
    model_device = get_model_device(model)
    return {
        k: v.to(model_device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }


def score_choices_by_loglikelihood(row, normalize_by_length=True):
    """Return one log-likelihood score per answer letter for a single VQA row."""
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    prefix_text = build_prompt(row, include_answer=False)
    num_choices = len(row["choices"])

    candidate_texts = [
        f"{prefix_text} {CHOICE_LETTERS[i]}"
        for i in range(num_choices)
    ]

    inputs = processor(
        text=candidate_texts,
        images=[image] * num_choices,
        return_tensors="pt",
        padding=True,
    )
    prefix_inputs = processor(
        text=[prefix_text],
        images=[image],
        return_tensors="pt",
        padding=False,
    )

    prefix_len = int(prefix_inputs["input_ids"].shape[1])
    inputs = move_to_model_device(inputs, model)
    input_ids = inputs["input_ids"]
    attention_mask = inputs.get("attention_mask")

    with torch.inference_mode():
        outputs = model(**inputs)

    log_probs = F.log_softmax(outputs.logits[:, :-1, :], dim=-1)
    target_ids = input_ids[:, 1:]
    token_log_probs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)

    scores = []
    token_counts = []
    start = max(prefix_len - 1, 0)

    for choice_idx in range(num_choices):
        if attention_mask is not None:
            seq_len = int(attention_mask[choice_idx].sum().item())
        else:
            seq_len = int(input_ids.shape[1])

        # After the shift, position j predicts token j+1 in the original sequence; answer tokens start at prefix_len.
        end = max(seq_len - 1, start + 1)
        answer_token_log_probs = token_log_probs[choice_idx, start:end]

        score = answer_token_log_probs.mean() if normalize_by_length else answer_token_log_probs.sum()
        scores.append(float(score.item()))
        token_counts.append(int(answer_token_log_probs.numel()))

    return scores, token_counts


def predict_row_by_loglikelihood(row, normalize_by_length=True):
    scores, token_counts = score_choices_by_loglikelihood(
        row,
        normalize_by_length=normalize_by_length,
    )
    pred_idx = int(np.argmax(scores))
    sorted_scores = sorted(scores, reverse=True)
    margin = float(sorted_scores[0] - sorted_scores[1]) if len(sorted_scores) > 1 else np.nan
    return pred_idx, scores, token_counts, margin


def evaluate_with_loglikelihood(df, desc="Evaluating with log-likelihood"):
    rows = []

    for i in tqdm(range(len(df)), desc=desc):
        row = df.iloc[i]
        pred_idx, scores, token_counts, margin = predict_row_by_loglikelihood(row)
        gt = int(row["answer"])

        result = {
            "id": row["id"],
            "pred": pred_idx,
            "gt": gt,
            "pred_letter": CHOICE_LETTERS[pred_idx],
            "gt_letter": CHOICE_LETTERS[gt],
            "correct": int(pred_idx == gt),
            "score_margin": margin,
            "question": row["question"],
        }

        for col in ANALYSIS_COLUMNS:
            if col in row.index:
                result[col] = row[col]

        for choice_idx, score in enumerate(scores):
            result[f"score_{CHOICE_LETTERS[choice_idx]}"] = score
            result[f"tokens_{CHOICE_LETTERS[choice_idx]}"] = token_counts[choice_idx]

        rows.append(result)

    return pd.DataFrame(rows)


val_result_df = evaluate_with_loglikelihood(
    val_df,
    desc="Evaluating trained model on val with log-likelihood",
)

val_acc = val_result_df["correct"].mean()
print(f"Validation accuracy (log-likelihood): {val_acc:.4f} ({val_result_df['correct'].sum()}/{len(val_result_df)})")
print("Prediction distribution:")
display(val_result_df["pred_letter"].value_counts().sort_index().rename("count").to_frame())
display(val_result_df.head())

In [ ]:
# Validation set error analysis

analysis_df = val_result_df.copy()


def summarize_accuracy_by(df, column, min_count=1):
    summary = (
        df.groupby(column, dropna=False)
        .agg(
            n=("correct", "size"),
            accuracy=("correct", "mean"),
            errors=("correct", lambda x: int((1 - x).sum())),
        )
        .reset_index()
    )
    summary = summary[summary["n"] >= min_count]
    return summary.sort_values(["accuracy", "n"], ascending=[True, False])


print("Overall accuracy:", f"{analysis_df['correct'].mean():.4f}")

print("\nAccuracy by number of choices:")
display(summarize_accuracy_by(analysis_df, "num_choices"))

for col in ["subject", "grade", "topic", "category", "skill"]:
    if col in analysis_df.columns:
        print(f"\nWorst groups by {col}:")
        display(summarize_accuracy_by(analysis_df, col, min_count=5).head(10))

print("\nPrediction vs ground truth distribution:")
distribution_df = pd.DataFrame({
    "pred_count": analysis_df["pred_letter"].value_counts().sort_index(),
    "gt_count": analysis_df["gt_letter"].value_counts().sort_index(),
}).fillna(0).astype(int)
display(distribution_df)

print("\nConfusion matrix:")
display(pd.crosstab(
    analysis_df["gt_letter"],
    analysis_df["pred_letter"],
    rownames=["ground_truth"],
    colnames=["prediction"],
))

print("\nHigh-confidence mistakes (small margin means the model was unsure):")
wrong_examples = (
    analysis_df[analysis_df["correct"] == 0]
    .sort_values("score_margin", ascending=False)
    .head(20)
)
display(wrong_examples[[
    "id",
    "gt_letter",
    "pred_letter",
    "score_margin",
    "num_choices",
    "subject",
    "topic",
    "category",
    "skill",
    "question",
]])

In [ ]:
# Generate submission file on the test set using multiple-choice log-likelihood

pred_rows = []

for i in tqdm(range(len(test_df)), desc="Generating submission with log-likelihood"):
    row = test_df.iloc[i]
    pred_idx, scores, token_counts, margin = predict_row_by_loglikelihood(row)

    pred_rows.append({
        "id": row["id"],
        "answer": int(pred_idx),
    })

submission_df = pd.DataFrame(pred_rows)
submission_df.to_csv("submission.csv", index=False)

print(submission_df.head())
print(submission_df.shape)
print("Saved submission.csv with log-likelihood predictions.")

In [ ]:

SAVE_SUB_PATH = "/content/drive/MyDrive/final_dl/submission.csv"
shutil.copy("submission.csv", SAVE_SUB_PATH)

print("Saved submission to:", SAVE_SUB_PATH)